In [2]:
# Install required packages (uncomment if needed)
!pip install opencv-python mediapipe numpy


In [1]:
import cv2



In [2]:
import mediapipe as mp


In [3]:
import os


In [4]:
import numpy as np


In [5]:
from datetime import datetime


In [6]:
import csv

In [35]:
# Initialize MediaPipe Face Detection
mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

In [50]:
# Paths
KNOWN_FACES_DIR = 'known_faces'
ATTENDANCE_FILE = 'attendance.csv'

marked_today = set()
today_date = datetime.now().strftime('%Y-%m-%d')

# Load names already marked today
if os.path.exists(ATTENDANCE_FILE):
    with open(ATTENDANCE_FILE, 'r', newline='') as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) >= 2:
                name = row[0]
                timestamp = row[1]
                if timestamp.startswith(today_date):
                    marked_today.add(name)


In [51]:
# Load known faces
known_faces = []
known_names = []

In [52]:
for filename in os.listdir(KNOWN_FACES_DIR):
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        image_path = os.path.join(KNOWN_FACES_DIR, filename)
        image = cv2.imread(image_path)
        if image is not None:
            gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            resized_image = cv2.resize(gray_image, (100, 100))
            equalized_image = cv2.equalizeHist(resized_image)
            known_faces.append(equalized_image)
            known_names.append(os.path.splitext(filename)[0])

print(f"Loaded {len(known_faces)} known faces.")

Loaded 1 known faces.


In [53]:
# Compare face images
def compare_faces(face1, face2):
    face1 = cv2.resize(face1, (100, 100))
    face1 = cv2.equalizeHist(face1)
    diff = cv2.absdiff(face1, face2)
    score = np.sum(diff)
    return score

In [54]:
# Mark attendance
# def mark_attendance(name):
#     now = datetime.now()
#     time_string = now.strftime('%Y-%m-%d %H:%M:%S')
#     with open(ATTENDANCE_FILE, 'a', newline='') as f:
#         writer = csv.writer(f)
#         writer.writerow([name, time_string])
#     print(f"Attendance marked for {name} at {time_string}")

def mark_attendance(name):
    if name not in marked_today:
        now = datetime.now()
        time_string = now.strftime('%Y-%m-%d %H:%M:%S')

        with open(ATTENDANCE_FILE, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([name, time_string])

        marked_today.add(name)
        print(f"Attendance marked for {name} at {time_string}")
    else:
        print(f"{name} is already marked present today.")

In [55]:
# Mark attendance (updated to prevent duplicates)
def mark_attendance(name):
    # Read existing names from the CSV to avoid duplicates
    try:
        with open(ATTENDANCE_FILE, 'r') as f:
            existing_entries = f.readlines()
        recorded_names = [line.split(',')[0] for line in existing_entries]
    except FileNotFoundError:
        # If file doesn't exist yet, no one is recorded
        recorded_names = []

    if name not in recorded_names:
        now = datetime.now()
        time_string = now.strftime('%Y-%m-%d %H:%M:%S')
        with open(ATTENDANCE_FILE, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([name, time_string])
        print(f"Attendance marked for {name} at {time_string}") 
    else:
        print(f"{name} is already marked present.")


In [56]:
# Start webcam
cap = cv2.VideoCapture(0)

with mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5) as face_detection:
    print("Starting face recognition. Press 'q' to quit.")
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_detection.process(rgb_frame)

        if results.detections:
            for detection in results.detections:
                bboxC = detection.location_data.relative_bounding_box
                h, w, _ = frame.shape
                x1 = int(bboxC.xmin * w)
                y1 = int(bboxC.ymin * h)
                x2 = x1 + int(bboxC.width * w)
                y2 = y1 + int(bboxC.height * h)

                face_img = frame[y1:y2, x1:x2]
                if face_img.size == 0 or face_img.shape[0] < 50 or face_img.shape[1] < 50:
                    continue
                gray_face = cv2.cvtColor(face_img, cv2.COLOR_BGR2GRAY)

                best_score = float('inf')
                best_match = None
                for known_face, name in zip(known_faces, known_names):
                    score = compare_faces(gray_face, known_face)
                    # print(f"Matching score with {name}: {score}")
                    if score < best_score:
                        best_score = score
                        best_match = name

                if best_score < 800000:
                    mark_attendance(best_match)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, best_match, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
                else:
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame,"Uknown", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

        cv2.imshow('MediaPipe Face Recognition Attendance', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


Starting face recognition. Press 'q' to quit.
Attendance marked for sasika7 at 2026-03-10 19:35:28
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked present.
sasika7 is already marked 